<div dir="rtl">

# 🚀 07 - End-to-End Complete Vector Store & RAG Pipeline (من الصفر للاحتراف)

## المسار الكامل لمنظومة الـ RAG (Retrieval-Augmented Generation):
هذا الكراس يمثل **المحطة الختامية المتكاملة**؛ حيث سنبدأ من قراءة البيانات الخام وصولاً إلى توليد الإجابات الدقيقة باستخدام النموذج اللغوي مع الاستشهاد بالمصادر.

```mermaid
graph LR
    A[📄 Raw Documents] --> B[✂️ Text Splitting]
    B --> C[🧠 Embedding Model]
    C --> D[🗄️ Vector Store FAISS]
    D --> E[🎯 MMR Retriever]
    E --> F[📝 LCEL Prompt Chain]
    F --> G[🤖 Grounded Answer]
```

---

### 🎯 ما سنبنيه في هذا الكراس خطوة بخطوة:
1. **قراءة وتحميل مستند المعرفة** من `data/sample.txt`.
2. **تقسيم المستند إلى Chunks متوازنة** عبر `RecursiveCharacterTextSplitter`.
3. **توليد التضمينات** باستخدام `HuggingFaceEmbeddings`.
4. **بناء وتخزين الفهرس محلياً** في مستودع `FAISS` مع إمكانية الحفظ والاسترجاع.
5. **تكوين مُسترجِع ذكي (MMR Retriever)** يضمن التنوع ويمنع تكرار المعلومات.
6. **بناء سلسلة الـ RAG الكاملة عبر LCEL** مع صياغة Prompt موجه لمنع الهلوسة.
7. **إجراء اختبارات شاملة**: أسئلة مباشرة، أسئلة مقارنة، استعلامات بالعربية، وأسئلة خارج النطاق.

</div>


<div dir="rtl">

### 1️⃣ الخطوة الأولى: تحميل البيانات الخام (Data Ingestion)

</div>


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# تحميل متغيرات البيئة
load_dotenv(find_dotenv())

# قراءة الملف التجريبي
DATA_DIR = Path("data") if Path("data").exists() else Path("../../data") if Path("../../data").exists() else Path("../data")
sample_file = DATA_DIR / "sample.txt"

loader = TextLoader(str(sample_file), encoding="utf-8")
raw_documents = loader.load()

print(f"✅ [1/7] تم تحميل المستند الأصلي: {len(raw_documents[0].page_content):,} حرف.")


<div dir="rtl">

### 2️⃣ الخطوة الثانية: تقسيم النص إلى قطع سياقية (Text Splitting)
نستخدم `chunk_size=500` و `chunk_overlap=100` وفقاً لأفضل الممارسات.

</div>


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["

", "
", ". ", " ", ""]
)

chunks = splitter.split_documents(raw_documents)

# ترقيم القطع في الميتاداتا لسهولة التتبع
for i, chunk in enumerate(chunks, 1):
    chunk.metadata["chunk_id"] = i

print(f"✅ [2/7] تم تقسيم المستند إلى {len(chunks)} قطعة.")
print(f"عينة من القطعة #1:\n{chunks[0].page_content[:150]}...")


<div dir="rtl">

### 3️⃣ الخطوة الثالثة: تجهيز نموذج التضمين (Embeddings Model)

</div>


In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("✅ [3/7] تم تحميل نموذج التضمين بنجاح!")


<div dir="rtl">

### 4️⃣ الخطوة الرابعة: إنشاء مستودع المتجهات (Vector Store) وحفظه محلياً

</div>


In [ ]:
# بناء فهرس FAISS
vector_store = FAISS.from_documents(chunks, embeddings)

# حفظ الفهرس محلياً على القرص
faiss_save_dir = DATA_DIR / "faiss_e2e_index"
os.makedirs(faiss_save_dir, exist_ok=True)
vector_store.save_local(str(faiss_save_dir))

print(f"✅ [4/7] تم إنشاء وحفظ مستودع FAISS بنجاح في: {faiss_save_dir.resolve()}")


<div dir="rtl">

### 5️⃣ الخطوة الخامسة: تكوين الـ Retriever مع استراتيجية MMR
استخدام **Maximal Marginal Relevance** لجلب مقاطع متنوعة غير مكررة في المعنى.

</div>


In [ ]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,          # عدد المستندات النهائية المراد إرسالها للنموذج
        "fetch_k": 8,    # عدد المرشحات الأولية للفحص
        "lambda_mult": 0.5 # التوازن المثالي بين الصلة والتنوع
    }
)

print("✅ [5/7] تم تجهيز الـ MMR Retriever بنجاح!")


<div dir="rtl">

### 6️⃣ الخطوة السادسة: بناء سلسلة الـ RAG باستخدام LCEL والقالب الذكي

</div>


In [ ]:
# دالة مساعدة لتنسيق المستندات المسترجعة مع معرفاتها
def format_docs_with_sources(docs):
    formatted = []
    for doc in docs:
        cid = doc.metadata.get("chunk_id", "?")
        formatted.append(f"[مصدر/Chunk #{cid}]:\n{doc.page_content}")
    return "\n\n".join(formatted)

# صياغة قالب الـ Prompt الموجه
prompt_template = """أنت مساعد ذكاء اصطناعي محترف ودقيق. أجب عن السؤال التالي بالاعتماد الكامل على السياق المرفق فقط.
إذا لم تجد الإجابة في السياق، اذكر بوضوح أن المعلومة غير متوفرة في الوثائق دون أي تخمين أو اختلاق.

السياق المتاح:
{context}

السؤال المطروح:
{question}

الإجابة الموثقة مع ذكر أرقام المقاطع المستند إليها:"""

prompt = PromptTemplate.from_template(prompt_template)

# محاكاة خط الأنابيب (أو ربطه مع Groq / OpenAI إذا توفر مفتاح الـ API)
groq_key = os.getenv("GROQ_API_KEY")

if groq_key and not groq_key.startswith("your_"):
    from langchain_groq import ChatGroq
    llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.1)
    rag_chain = (
        {"context": retriever | format_docs_with_sources, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    print("✅ [6/7] تم بناء RAG Chain كاملة مربوطة بنموذج Groq (Llama 3.1) عبر LCEL!")
else:
    # سلسلة تقوم بتجهيز وإخراج الـ Prompt المهيأ مع السياق المسترجع
    rag_chain = (
        {"context": retriever | format_docs_with_sources, "question": RunnablePassthrough()}
        | prompt
        | StrOutputParser()
    )
    print("✅ [6/7] تم بناء LCEL RAG Chain (تجهيز السياق والـ Prompt المسترجع بنجاح)!")


<div dir="rtl">

### 7️⃣ الخطوة السابعة: الاختبار العملي الشامل للاستعلامات المختلفة

</div>


In [ ]:
test_questions = [
    "What is the recommended chunk_size and chunk_overlap for general Q&A?",
    "Which vector database is built with Rust and what are its strengths?",
    "ما هي فوائد تقنية RAG في تقليل هلوسة النماذج اللغوية؟"
]

print("="*70)
print("🎯 نتائج تشغيل الـ RAG Pipeline على الأسئلة التجريبية:")
print("="*70)

for i, q in enumerate(test_questions, 1):
    print(f"\n❓ [سؤال {i}]: {q}\n")
    response = rag_chain.invoke(q)
    print("💡 [النتيجة / الإجابة المسترجعة]:")
    print(response)
    print("-" * 60)

print("\n🎉 تهانينا! لقد اكتمل بناء وتشغيل منظومة الـ RAG والـ Vector Store بالكامل من الصفر وحتى خط النهاية!")
